# 05. Knowledge Distillation: Logit, Feature, and MiniLLM

- **Category:** 15_compression_peft_and_speculative_inference / 01_pruning_sparsity_lowrank_and_distillation
- **Lineage & Citations:** Hinton et al. (2015, *Distilling the Knowledge in a Neural Network*, NeurIPS-W); Romero et al. (2014, *FitNets: Hints for Thin Deep Nets*, ICLR); Gu et al. (2023, *MiniLLM: Knowledge Distillation of Large Language Models*, ICLR).
- **Purpose:** Formulate teacher-student compression across output and intermediate representations: Hinton logit-based distillation with temperature scaling, FitNets hidden state feature alignment via linear projectors, and MiniLLM Reverse KL divergence for mode-seeking generative language modeling.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Low-Rank Decomposition: SVD, ASVD, Tucker, Monarch, and Tensor Train](04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Adaptive Compute, Early Exit, Token Merging, and FrugalGPT Cascades](../02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply forward KL divergence $D_{\text{KL}}(P_T \parallel P_S)$ to compress autoregressive generative LLMs (forward KL is zero-avoiding and causes hallucination by forcing student coverage over improbable teacher tails; use Reverse KL instead); do NOT scale temperature $T$ without multiplying KL loss by $T^2$ (gradient magnitudes vanish as $T$ increases).

## Mathematical Foundations & Governing Equations

1. **Hinton Temperature-Scaled Softmax & Loss**:
   $$p_i^T = \frac{\exp(z_i^T / T)}{\sum_j \exp(z_j^T / T)}, \quad \mathcal{L}_{\text{KD}} = (1 - \alpha) \mathcal{L}_{\text{CE}}(y, p^S) + \alpha T^2 D_{\text{KL}}(p^T \parallel p^S)$$
2. **FitNets Intermediate Feature Alignment**:
   $$\mathcal{L}_{\text{feat}} = \frac{1}{2 B} \|h^S W_{\text{proj}} - h^T\|_F^2$$
3. **Reverse KL Divergence (MiniLLM)**:
   $$D_{\text{KL}}(p^S \parallel p^T) = \sum_i p_i^S \left( \log p_i^S - \log p_i^T \right)$$

### Asymptotic Complexity
- Logit Distillation: $\mathcal{O}(B \cdot V)$ probability reduction and divergence computation.
- Feature Distillation: $\mathcal{O}(B \cdot d_S \cdot d_T)$ linear projection and MSE.

In [ ]:
import sys, time
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} golden tensors from {WEIGHTS_PATH.name}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def hinton_kd_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
    p_s_log = F.log_softmax(student_logits / temperature, dim=-1)
    p_t = F.softmax(teacher_logits / temperature, dim=-1)
    p_t_log = F.log_softmax(teacher_logits / temperature, dim=-1)
    kl_div = torch.sum(p_t * (p_t_log - p_s_log), dim=-1).mean()
    ce_loss = F.cross_entropy(student_logits, labels)
    return (1.0 - alpha) * ce_loss + alpha * (temperature**2) * kl_div, kl_div

def feature_distillation_loss(student_feat, teacher_feat, proj_weight):
    return F.mse_loss(student_feat @ proj_weight, teacher_feat)

def reverse_kl_loss(student_logits, teacher_logits, temperature=2.0):
    p_s = F.softmax(student_logits / temperature, dim=-1)
    p_s_log = F.log_softmax(student_logits / temperature, dim=-1)
    p_t_log = F.log_softmax(teacher_logits / temperature, dim=-1)
    return torch.sum(p_s * (p_s_log - p_t_log), dim=-1).mean()

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

# 2. Hinton KD loss parity
total_kd, fwd_kl = hinton_kd_loss(weights["student_logits"], weights["teacher_logits"], weights["labels"], temperature=2.0, alpha=0.5)
assert abs(float(total_kd) - float(weights["loss_metrics"][0])) < 1e-5
assert abs(float(fwd_kl) - float(weights["loss_metrics"][1])) < 1e-5

# 3. Reverse KL divergence parity
rev_kl = reverse_kl_loss(weights["student_logits"], weights["teacher_logits"], temperature=2.0)
assert abs(float(rev_kl) - float(weights["loss_metrics"][2])) < 1e-5

# 4. Feature map distillation parity
feat_loss = feature_distillation_loss(weights["student_feat"], weights["teacher_feat"], weights["proj_weight"])
assert abs(float(feat_loss) - float(weights["loss_metrics"][3])) < 1e-5
print("All Knowledge Distillation implementations verified against seed-42 golden tensors!")

In [ ]:
# Empirical evaluation: Forward vs. Reverse KL Discrepancy
print(f"[1] Forward KL D_KL(P_T || P_S): {float(fwd_kl):.4f} (Mode-Covering / Mean-Seeking)")
print(f"[2] Reverse KL D_KL(P_S || P_T): {float(rev_kl):.4f} (Mode-Seeking / Precision-Oriented)")
print(f"[3] FitNets Feature MSE:         {float(feat_loss):.4f}")
print(f"[4] Total Hinton KD Loss:        {float(total_kd):.4f}")
assert fwd_kl > 0 and rev_kl > 0, "Divergences must be strictly positive"
print("Asymmetry confirmed: Forward KL != Reverse KL across probability distributions.")

In [ ]:
# Pathological Misuse: Temperature Scaling Gradient Collapse
# Distilling at temperature T without multiplying loss by T^2 causes gradient scale to decay as 1/T^2!
T_high = 10.0
s1 = weights["student_logits"].clone().requires_grad_(True)
p_s1 = F.log_softmax(s1 / T_high, dim=-1)
p_t = F.softmax(weights["teacher_logits"] / T_high, dim=-1)
loss_unscaled = torch.sum(p_t * (p_t.log() - p_s1), dim=-1).mean()
loss_unscaled.backward()
grad_unscaled = float(torch.norm(s1.grad))

s2 = weights["student_logits"].clone().requires_grad_(True)
p_s2 = F.log_softmax(s2 / T_high, dim=-1)
loss_scaled = (T_high ** 2) * torch.sum(p_t * (p_t.log() - p_s2), dim=-1).mean()
loss_scaled.backward()
grad_scaled = float(torch.norm(s2.grad))
print(f"Gradient norm without T^2 scaling: {grad_unscaled:.6f}")
print(f"Gradient norm with T^2 scaling:    {grad_scaled:.6f} ({grad_scaled/grad_unscaled:.0f}x restored!)")
assert grad_scaled > 50.0 * grad_unscaled, "T^2 scaling must restore gradient magnitude!"
print("Pathology demonstrated: Omitting T^2 scaling causes severe gradient starvation at high temperatures.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Dark Knowledge**: Softening logits with temperature $T$ uncovers inter-class semantic structures that enable small students to emulate large teachers.
2. **Mode-Covering vs Mode-Seeking**: Forward KL ($D_{\text{KL}}(P_T \parallel P_S)$) causes hallucinations in generative LLMs; Reverse KL ($D_{\text{KL}}(P_S \parallel P_T)$, MiniLLM) constrains students to sharp teacher modes.
3. **T^2 Compensation**: The $T^2$ loss multiplier is mathematically mandatory to preserve gradient scale as temperature softening attenuates soft label differences.

---
### Cross-Domain Generalisation Matrix
- **Reinforcement Learning**: Reverse KL distillation is mathematically isomorphic to PPO policy entropy regularization against a reference policy.
- **Variational Inference**: The evidence lower bound (ELBO) maximizes $-D_{\text{KL}}(q_\phi(z|x) \parallel p(z|x))$, operating under the exact same reverse-KL mode-seeking dynamic.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_knowledge_distillation_logit_feature_and_minillm.md](../../../markdown/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.md)
- **PyTorch 2.x Module:** [05_knowledge_distillation_logit_feature_and_minillm.py](../../../pytorch2/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.py)
- **Functional JAX Module:** [05_knowledge_distillation_logit_feature_and_minillm.py](../../../jax/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.py)
- **Native Rust Candle Track:** [05_knowledge_distillation_logit_feature_and_minillm.rs](../../../candle/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.rs)
- **Hardware Kernel (CPU / WGSL):** [05_knowledge_distillation_logit_feature_and_minillm.rs](../../../kernels/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.rs)
- **Golden SafeTensors:** [05_knowledge_distillation_logit_feature_and_minillm.safetensors](../../../weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/05_knowledge_distillation_logit_feature_and_minillm.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Low Rank Decomposition SVD Asvd Tucker Monarch and Tt](04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 01. Adaptive Compute Early Exit Token Merging and Frugalgpt Cascades](../02_adaptive_kv_speculative_and_merging/01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.ipynb) &rarr;